# ENSy on Google Colab

Exploratory runs and the sensitivity analysis. **The thesis numbers come from the GitHub Actions run, not from this notebook** (Colab's preinstalled packages drift).

1. Add your Hugging Face token as a Colab secret named `HF_TOKEN` (key icon in the left bar) and enable notebook access.
2. Set `COMMIT` below to the commit you want to reproduce.
3. Run all cells (CPU runtime is enough).

In [ ]:
COMMIT = "ensy-v2.1.3"  # branch, tag or commit SHA
REPO = "https://github.com/sans-script/manifold.git"  # private repo: use a token URL or make it public

!git clone --quiet {REPO} manifold && cd manifold && git checkout --quiet {COMMIT} && git log --oneline -1

In [ ]:
%cd manifold
!pip install --quiet torch==2.6.0 --index-url https://download.pytorch.org/whl/cpu
!pip install --quiet -r requirements.txt

In [ ]:
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")  # never print it

In [ ]:
!python -m pytest tests/test_coranking.py -q -p no:playwright

In [ ]:
!python -m ensy.experiment --out colab-results
!python -m ensy.experiment --sensitivity --out colab-results

In [ ]:
!for f in colab-results/main_*.json; do python -m ensy.compare results/$(basename $f) $f; done

In [ ]:
import json, glob
import matplotlib.pyplot as plt

res = json.load(open(sorted(glob.glob("colab-results/main_*.json"))[-1]))
models = sorted({c["model"] for c in res["configs"]})
fig, axes = plt.subplots(1, len(models), figsize=(4 * len(models), 3.2), sharey=True)
for ax, m in zip(axes, models):
    for c in (c for c in res["configs"] if c["model"] == m):
        ks = [p["K"] for p in c["curves"]]
        ax.plot(ks, [p["trustworthiness"] for p in c["curves"]], marker="o", label=c["algorithm"])
    ax.set_title(m); ax.set_xlabel("K")
axes[0].set_ylabel("Trustworthiness $M_T(K)$"); axes[-1].legend()
plt.tight_layout()